In [40]:
import sys
import os
import torch
from transformers import AutoProcessor, AutoTokenizer

In [2]:
module_path = "/home/ubuntu/Shree_FYP/train/stage2/models"

In [3]:
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
from final_latent_student import LatentStudent
from final_verbalizer import Verbalizer

In [44]:
processor = AutoProcessor.from_pretrained("/home/ubuntu/Shree_FYP/data/stage1_unsloth")

[transformers] The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [45]:
student = LatentStudent(model_name="/home/ubuntu/Shree_FYP/data/stage1_unsloth")

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [46]:
student.vlm.config.text_config.num_hidden_layers

32

In [47]:
student.to("cuda")

batch_size = 1
seq_len = 64
device = next(student.parameters()).device

In [48]:
input_ids = torch.randint(100, 5000, (batch_size, seq_len), device=device)
attention_mask = torch.ones_like(input_ids, device=device)
pixel_values = None          # Skip vision encoder for quick test
image_grid_thw = None

# 3. Shape & Structure Test (Inference Mode)
print("\n🔍 Running shape verification (no_grad)...")
student.eval()
with torch.no_grad():
    latents, spatial_hidden, waypoints = student.generate_latents(
        input_ids=input_ids,
        pixel_values=pixel_values,
        image_grid_thw=image_grid_thw,
        attention_mask=attention_mask,
    )

# Verify outputs
assert len(latents) == 6, f"❌ Expected 6 latents, got {len(latents)}"
for i, z in enumerate(latents):
    assert z.shape == (batch_size, 2560), f"❌ z_{i+1} shape mismatch: {z.shape}"
    print(f"  ✅ z_{i+1} shape: {z.shape}")

assert spatial_hidden.shape == (batch_size, 5, 2560), f"❌ spatial_hidden mismatch: {spatial_hidden.shape}"
print(f"  ✅ spatial_hidden shape: {spatial_hidden.shape}")

assert waypoints.shape == (batch_size, 5, 2), f"❌ waypoints mismatch: {waypoints.shape}"
print(f"  ✅ waypoints shape: {waypoints.shape}")
assert waypoints.min() >= 0.0 and waypoints.max() <= 1.0, "❌ Waypoints out of [0,1] range!"
print(f"  ✅ Waypoints range: [{waypoints.min().item():.3f}, {waypoints.max().item():.3f}]")


🔍 Running shape verification (no_grad)...
  ✅ z_1 shape: torch.Size([1, 2560])
  ✅ z_2 shape: torch.Size([1, 2560])
  ✅ z_3 shape: torch.Size([1, 2560])
  ✅ z_4 shape: torch.Size([1, 2560])
  ✅ z_5 shape: torch.Size([1, 2560])
  ✅ z_6 shape: torch.Size([1, 2560])
  ✅ spatial_hidden shape: torch.Size([1, 5, 2560])
  ✅ waypoints shape: torch.Size([1, 5, 2])
  ✅ Waypoints range: [0.463, 0.559]


In [49]:
# 4. Gradient Flow Test (Training Mode)
print("\n🔗 Testing gradient flow...")
student.train()
latents, spatial_hidden, waypoints = student.generate_latents(
    input_ids, pixel_values, image_grid_thw, attention_mask
)

# Dummy scalar loss to trigger backward
loss = sum(l.sum() for l in latents) + spatial_hidden.sum() + waypoints.sum()
loss.backward()

# Check if LoRA & Spatial params received gradients
grad_count = 0
for name, p in student.named_parameters():
    if p.requires_grad and p.grad is not None:
        grad_count += 1

print(f"  ✅ {grad_count} parameters received gradients.")
assert grad_count > 0, "❌ No gradients flowed! Check use_cache=False & LoRA wrapping."
print("\n🎉 All tests passed. Model is ready for Stage 2 training.")


🔗 Testing gradient flow...
  ✅ 503 parameters received gradients.

🎉 All tests passed. Model is ready for Stage 2 training.


## Test with textual input (prompt)

In [50]:
prompt = "LLM is a large language model right?"

In [51]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": f"{prompt}",
    }
]

In [52]:
# Apply chat template to get the raw string
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"\n📝 Generated Prompt:\n{text}")


📝 Generated Prompt:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
LLM is a large language model right?<|im_end|>
<|im_start|>assistant
<think>

</think>




In [53]:
# Tokenize to get input_ids and attention_mask
inputs = processor(text=[text], return_tensors="pt", padding=True)
input_ids = inputs.input_ids.to("cuda")
attention_mask = inputs.attention_mask.to("cuda")

In [54]:
print(f"🔢 Input IDs shape: {input_ids.shape}")
print(f"🎭 Attention Mask shape: {attention_mask.shape}")

🔢 Input IDs shape: torch.Size([1, 32])
🎭 Attention Mask shape: torch.Size([1, 32])


In [55]:
# 4. Run Generation (No Image for this test)
print("\n🚀 Running generate_latents with real text...")
student.eval()
with torch.no_grad():
    latents, spatial_hidden, waypoints = student.generate_latents(
        input_ids=input_ids,
        pixel_values=None,       # No image
        image_grid_thw=None,     # No image grid
        attention_mask=attention_mask
    )

# 5. Verify Outputs
print("\n✅ Verification Results:")
assert len(latents) == 6, f"❌ Expected 6 latents, got {len(latents)}"
for i, z in enumerate(latents):
    assert z.shape == (1, 2560), f"❌ z_{i+1} shape mismatch: {z.shape}"
    print(f"  ✅ z_{i+1} shape: {z.shape} (dtype: {z.dtype})")

assert spatial_hidden.shape == (1, 5, 2560), f"❌ spatial_hidden mismatch: {spatial_hidden.shape}"
print(f"  ✅ spatial_hidden shape: {spatial_hidden.shape}")

assert waypoints.shape == (1, 5, 2), f"❌ waypoints mismatch: {waypoints.shape}"
print(f"  ✅ waypoints shape: {waypoints.shape}")

# Check waypoint range (should be [0, 1] due to Sigmoid)
wp_min = waypoints.min().item()
wp_max = waypoints.max().item()
print(f"  ✅ Waypoints range: [{wp_min:.4f}, {wp_max:.4f}]")
assert 0.0 <= wp_min and wp_max <= 1.0, "❌ Waypoints out of [0,1] range!"

print("\n🎉 Success! The model correctly processes real textual prompts.")


🚀 Running generate_latents with real text...

✅ Verification Results:
  ✅ z_1 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_2 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_3 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_4 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_5 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_6 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ spatial_hidden shape: torch.Size([1, 5, 2560])
  ✅ waypoints shape: torch.Size([1, 5, 2])
  ✅ Waypoints range: [0.3555, 0.5625]

🎉 Success! The model correctly processes real textual prompts.


In [56]:
del student

In [57]:
import gc

In [58]:
torch.cuda.empty_cache()
gc.collect()

7181

In [59]:
type(latents[0])

torch.Tensor

## Send Latent Vectors to Verbalizer

In [60]:
# 1. Initialize Verbalizer
print("📦 Loading Verbalizer...")
v = Verbalizer(model_name="unsloth/Qwen3.5-0.8B", student_hidden=2560)

# 2. Load tokenizer (Qwen3.5 requires trust_remote_code)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-0.8B", trust_remote_code=True)

📦 Loading Verbalizer...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [61]:
# 2. YOUR chat-formatted prompt
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "LLM is a large language model right?"}
]

# 3. Apply Qwen's chat template & tokenize
# add_generation_prompt=True appends the assistant turn starter so the model knows it's time to generate
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
generated_ids = inputs.input_ids.clone()

In [62]:
# 4. YOUR latents from latent_student (List of 6 × [1, 2560])
# Replace with your actual list: student_latents = [...]
latents_ = Verbalizer.stack_latents(latents).to("cuda", dtype=torch.bfloat16)
print(f"✅ Latents shape: {latents_.shape} | dtype: {latents_.dtype}")

✅ Latents shape: torch.Size([1, 6, 2560]) | dtype: torch.bfloat16


In [63]:
# 5. Autoregressive generation conditioned on latents
max_new_tokens = 50
print("🚀 Generating verbalized response from latents + chat prompt...")
with torch.no_grad():
    for _ in range(max_new_tokens):
        attn_mask = torch.ones_like(generated_ids)
        logits, _ = v._lm_forward(generated_ids, attn_mask, latents_)
        
        # Greedy decode next token
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated_ids = torch.cat([generated_ids, next_token], dim=1)

# 6. Decode & inspect
# skip_special_tokens=False lets you see <|im_start|>/assistant tags for debugging
output = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
print("\n🤖 Verbalized Output:\n", "-"*60)
print(output)
print("-"*60)

🚀 Generating verbalized response from latents + chat prompt...

🤖 Verbalized Output:
 ------------------------------------------------------------
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
LLM is a large language model right?<|im_end|>
<|im_start|>assistant
<think>

</think>

Yes, **LLM stands for Large Language Model**, and it is a large-scale, deep-learning system developed by companies like Google, Meta, and Microsoft. LLMs are trained on vast amounts of text data from the internet, including books,
------------------------------------------------------------


---
### Testing Gate
---
### Test 1:
Latent Prompt: Hi there how are you?\
Verbalizer Prompt: LLM is a large language model right?\
Verbalizer Output: Yes, **LLM stands for Large Language Model** ...\

### Test 2:
Latent Prompt: LLM is a large language model right?\
Verbalizer Prompt: LLM is a large language model right?\
Verbalizer Output: Yes, **LLM stands for Large Language Model** ...\

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain how LLM works?<|im_end|>
<|im_start|>assistant
<think>

</think>

保护和保护和保护和保护和保护

How does GPT work?
enteredentereden

How many hearts does an octopus have?
ewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareeware
------------------------------------------------------------

describe the steps to solve a simple physics problem involving projectile motion
一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题
------------------------------------------------------------

Sally has 3 sisters, each of her sisters have 2 brothers, how many brother does Sally have?
лялилялилялилялилялилялилялилялилялилялилялилялилялилялилялилялилял

In [31]:
del v
torch.cuda.empty_cache()
gc.collect()

39640